# ANON4REID - CPU-Bound Dataset Generation (Kaggle)
This notebook generates the Level 1 (Gaussian Blur) and Level 4 (Canny Edge Silhouette) datasets for Market-1501. It is designed to run in a Kaggle environment where the Market-1501 dataset is attached to `/kaggle/input/` and outputs are written to `/kaggle/working/` so they can be downloaded as Zip archives.

In [ ]:
import os
import cv2
import shutil
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

print("Libraries loaded successfully.")

In [ ]:
def anonymize_blur(image: Image.Image) -> Image.Image:
    """
    Applies Gaussian Blur to the head region of the image based on spatial heuristics.
    Market-1501 images are relatively consistent bounding boxes of 64x128.
    """
    # Convert PIL Image to OpenCV format (RGB -> BGR)
    cv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
    h, w = cv_img.shape[:2]
    
    # Calculate the region of interest (top ~28% for the head/face)
    x1, x2 = int(w * 0.15), int(w * 0.85)
    y1, y2 = 0, int(h * 0.28)
    
    roi = cv_img[y1:y2, x1:x2]
    
    # Apply a heavy Gaussian blur to the ROI
    blurred_roi = cv2.GaussianBlur(roi, (15, 15), 30)
    
    # Place the blurred patch back onto the image
    cv_img[y1:y2, x1:x2] = blurred_roi
    
    # Convert OpenCV format back to PIL Image (BGR -> RGB)
    return Image.fromarray(cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB))

def anonymize_silhouette(image: Image.Image) -> Image.Image:
    """
    Applies Canny Edge Detection to strip all texturing/color and turn the 
    identity into a bare silhouette wireframe representing Level 4 anonymization.
    """
    # Convert PIL directly to Grayscale OpenCV numpy array
    cv_gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    
    # Apply Canny Edge Detection
    edges = cv2.Canny(cv_gray, 100, 200)
    
    # Convert back to 3-channel RGB
    edges_rgb = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    
    return Image.fromarray(edges_rgb)

print("Anonymization functions loaded.")

In [ ]:
# Path Configuration for Kaggle
# NOTE: Make sure the INPUT_DIR matches exactly what Kaggle names your dataset folder
INPUT_DIR = "/kaggle/input/datasets/rayiooo/reid_market-1501"
OUTPUT_DIR_BLUR = "/kaggle/working/Market-1501-Blur"
OUTPUT_DIR_EDGE = "/kaggle/working/Market-1501-Edge"

SPLITS = ["query", "bounding_box_test", "bounding_box_train"]

for d in [OUTPUT_DIR_BLUR, OUTPUT_DIR_EDGE]:
    os.makedirs(d, exist_ok=True)
    for split in SPLITS:
        os.makedirs(os.path.join(d, split), exist_ok=True)

print("Directories configured.")

In [ ]:
def process_dataset(input_dir, output_dir, proc_func, desc):
    for split in SPLITS:
        split_dir_in = os.path.join(input_dir, split)
        split_dir_out = os.path.join(output_dir, split)
        
        if not os.path.exists(split_dir_in):
            print(f"Skipping {split_dir_in} (not found)")
            continue
            
        images = [f for f in os.listdir(split_dir_in) if f.endswith('.jpg')]
        print(f"Processing {split} for {desc}: {len(images)} images")
        
        skipped = 0
        processed = 0
        
        for fname in tqdm(images, desc=f"{desc} - {split}"):
            in_path = os.path.join(split_dir_in, fname)
            out_path = os.path.join(split_dir_out, fname)
            
            # Skip if already processed to handle interruptions safely
            if os.path.exists(out_path):
                skipped += 1
                continue
                
            try:
                img = Image.open(in_path).convert("RGB")
                res = proc_func(img)
                res.save(out_path)
                processed += 1
            except Exception as e:
                print(f"Error on {fname}: {e}")
        
        print(f"  -> Processed: {processed}, Skipped: {skipped}")

# Process both pipelines sequentially
print("Starting Blur Pipeline (Level 1)...")
process_dataset(INPUT_DIR, OUTPUT_DIR_BLUR, anonymize_blur, "Blur")

print("\nStarting Edge Pipeline (Level 4)...")
process_dataset(INPUT_DIR, OUTPUT_DIR_EDGE, anonymize_silhouette, "Edge")

print("\nDataset Generation complete!")

In [ ]:
# Zipping the outputs so they can be easily downloaded from the Kaggle kernel data panel
print("Zipping Market-1501-Blur...")
shutil.make_archive("/kaggle/working/Market-1501-Blur", 'zip', OUTPUT_DIR_BLUR)

print("Zipping Market-1501-Edge...")
shutil.make_archive("/kaggle/working/Market-1501-Edge", 'zip', OUTPUT_DIR_EDGE)

print("Zipping Complete! You can now download the Output archives.")